# Chess Dataset — Data Acquisition & Parsing
### CMPS344 Applied Data Science — Phase 2
---
**Sources:**
- `data.pgn` — 50,000 chess games in standard PGN notation
- `data_uci.pgn` — Same games with moves in UCI coordinate notation
- `stockfish.csv` — Per-game Stockfish centipawn evaluations

**Goal:** Parse all three files, extract features, merge into a single clean DataFrame, and save as `games.csv`.

In [ ]:
# uncomment this line to install dependencies (first run only)
# %pip install python-chess requests

In [2]:
import os
import io
import re
import zipfile
import requests
import pandas as pd

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [27]:
# 1. Download Kaggle Datasets using Kaggle API magic commands``
# (This requires the kaggle API to be configured on your machine)
# os.environ["KAGGLE_API_TOKEN"] = "xxxxxxxxxxxxxxxxxxxxxxxxxxxx"

# from kaggle.api.kaggle_api_extended import KaggleApi
# api = KaggleApi()
# api.authenticate()
print("Downloading Kaggle Datasets...")
project_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
raw_dir = os.path.join(project_dir, "data/raw")
!kaggle competitions download -c finding-elo -p {raw_dir}
!kaggle datasets download mysarahmadbhat/online-chess-games -f chess_games.csv -p {raw_dir}

def unzip_all_in_dir(directory):
    extracted = True
    while extracted:
        extracted = False
        for file in os.listdir(directory):
            if file.endswith(".zip"):
                file_path = os.path.join(directory, file)
                print(f"Extracting {file}...")
                with zipfile.ZipFile(file_path, "r") as zip_ref:
                    zip_ref.extractall(directory)
                os.remove(file_path)
                extracted = True

print("\nUnzipping Kaggle files (including nested zips)...")
unzip_all_in_dir(raw_dir)


# 3. Download Lichess ECO Database from GitHub URL
print("\nDownloading ECO Database from GitHub...")
frames = []
for letter in list("abcde"):
    url = f"https://raw.githubusercontent.com/lichess-org/chess-openings/master/{letter}.tsv"
    resp = requests.get(url)
    df_letter = pd.read_csv(io.StringIO(resp.text), sep="\t")
    frames.append(df_letter)

df_eco = pd.concat(frames, ignore_index=True)
df_eco.to_csv(os.path.join(raw_dir, "eco_openings.csv"), index=False)
print("Data Acquisition Complete!")


  0%|          | 0.00/20.8M [00:00<?, ?B/s]
  5%|▍         | 1.00M/20.8M [00:00<00:13, 1.58MB/s]
 10%|▉         | 2.00M/20.8M [00:01<00:12, 1.60MB/s]
 14%|█▍        | 3.00M/20.8M [00:02<00:17, 1.04MB/s]
 19%|█▉        | 4.00M/20.8M [00:03<00:17, 984kB/s] 
 24%|██▍       | 5.00M/20.8M [00:06<00:28, 583kB/s]
 29%|██▉       | 6.00M/20.8M [00:08<00:23, 665kB/s]
 34%|███▎      | 7.00M/20.8M [00:08<00:16, 895kB/s]
 39%|███▊      | 8.00M/20.8M [00:09<00:13, 1.03MB/s]
 43%|████▎     | 9.00M/20.8M [00:12<00:21, 587kB/s] 
 48%|████▊     | 10.0M/20.8M [00:13<00:16, 672kB/s]
 53%|█████▎    | 11.0M/20.8M [00:14<00:12, 831kB/s]
 58%|█████▊    | 12.0M/20.8M [00:15<00:11, 833kB/s]
 63%|██████▎   | 13.0M/20.8M [00:17<00:12, 664kB/s]
 67%|██████▋   | 14.0M/20.8M [00:19<00:09, 716kB/s]
 72%|███████▏  | 15.0M/20.8M [00:19<00:06, 979kB/s]
 77%|███████▋  | 16.0M/20.8M [00:19<00:04, 1.08MB/s]
 82%|████████▏ | 17.0M/20.8M [00:21<00:04, 851kB/s] 
 87%|████████▋ | 18.0M/20.8M [00:22<00:03, 870kB/s]
 91%|██████

Dataset URL: https://www.kaggle.com/datasets/mysarahmadbhat/online-chess-games
License(s): CC0-1.0


Unzipping Kaggle files (including nested zips)...
Extracting chess_games.csv.zip...
Extracting finding-elo.zip...
Extracting data.pgn.zip...
Extracting data_uci.pgn.zip...
Extracting sampleSubmission.csv.zip...
Extracting stockfish.csv.zip...



  0%|          | 0.00/2.54M [00:00<?, ?B/s]
 39%|███▉      | 1.00M/2.54M [00:02<00:03, 441kB/s]
 79%|███████▉  | 2.00M/2.54M [00:04<00:01, 457kB/s]
100%|██████████| 2.54M/2.54M [00:04<00:00, 613kB/s]
100%|██████████| 2.54M/2.54M [00:04<00:00, 550kB/s]



Data Acquisition Complete!


In [32]:
import chess.pgn 
def parse_pgn(filepath: str) -> pd.DataFrame:
    """Extracts metadata and moves from standard PGN."""
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        while True:
            game = chess.pgn.read_game(f)
            if game is None: break
            
            # Skip games missing essential fields
            if "WhiteElo" not in game.headers or "BlackElo" not in game.headers: continue
                
            records.append({
                "event_id": int(game.headers.get("Event", 0)),
                "white_elo": int(game.headers.get("WhiteElo")),
                "black_elo": int(game.headers.get("BlackElo")),
                "result": game.headers.get("Result", "*"),
                "moves_san": " ".join([game.board().san(move) for move in game.mainline_moves()])
            })
    return pd.DataFrame(records)

def parse_uci(filepath: str) -> pd.DataFrame:
    """Extracts UCI move strings using regex."""
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    records = []
    for block in re.split(r"\n(?=\[Event )", content.strip()):
        event_match = re.search(r'\[Event "([^"]+)"\]', block)
        if event_match:
            moves = " ".join([line for line in block.split("\n") if line.strip() and not line.startswith("[")])
            records.append({
                "event_id": int(event_match.group(1)),
                "moves_uci": re.sub(r"\s*(1-0|0-1|1/2-1/2|\*)\s*$", "", moves).strip()
            })
    return pd.DataFrame(records)

intermediate_dir = os.path.join(project_dir, "data/intermediate")
print("Parsing UCI file...")
df_data_uci = parse_uci(raw_dir + "/data_uci.pgn")
df_data_uci.to_csv(os.path.join(intermediate_dir, "data_uci.csv"), index=False)
df_data_uci.head(2)

print("Parsing PGN file...")
df_data_pgn = parse_pgn(raw_dir + "/data.pgn")
df_data_pgn.to_csv(os.path.join(intermediate_dir, "data_pgn.csv"), index=False)
df_data_pgn.head(2)




print("Parsing Complete!")

Parsing UCI file...
Parsing PGN file...


AssertionError: san() and lan() expect move to be legal or null, but got c3d5 in rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1

In [ ]:
# Phase 1: Inner Join Kaggle Sources on 'event_id'
print("Loading Stockfish & Lichess CSVs...")
df_lichess_stockfish = pd.read_csv(intermediate_dir + "/lichess_stockfish.csv")
df_chess_games = pd.read_csv(raw_dir + "/chess_games.csv")

print("--- PHASE 1: KAGGLE MERGE ---")
print(f"Before -> PGN: {len(df_data_pgn)}, UCI: {len(df_data_uci)}, Stockfish: {len(df_lichess_stockfish)}")

# Rename Stockfish Event column to match event_id for merging
df_lichess_stockfish = df_lichess_stockfish.rename(columns={"Event": "event_id"})

df_kaggle_merged = (
    df_data_pgn
    .merge(df_data_uci, on="event_id", how="inner")
    .merge(df_lichess_stockfish, on="event_id", how="inner")
)
print(f"After Phase 1  -> Rows: {len(df_kaggle_merged)}")

# Phase 2: Harmonize Lichess columns to match Kaggle, then Concatenate
print("\n--- PHASE 2: LICHESS INTEGRATION ---")
print(f"Before -> Kaggle Data: {len(df_kaggle_merged)}, Lichess Data: {len(df_lichess)}")

# Tag source and align column names so they stack perfectly
df_kaggle_merged["source"] = "kaggle"
df_lichess["source"] = "lichess"

df_lichess = df_lichess.rename(columns={
    "game_id": "event_id",
    "white_rating": "white_elo",
    "black_rating": "black_elo",
    "winner": "result", # Note: Result formatting (1-0 vs White) handled later in transformation
    "moves": "moves_san"
})

# Concatenate (Union) the datasets together
df_final = pd.concat([df_kaggle_merged, df_lichess], ignore_index=True, sort=False)
df_final["event_id"] = range(1, len(df_final) + 1) # Reset IDs

df_final.to_csv("./data/intermi/final_chess_dataset.csv", index=False)

print(f"After Phase 2 -> Final Dataset Rows: {len(df_final)}")
print("\nTask 1 Data Pipeline Complete. Ready for Validation and Cleaning.")